# Compute residual norm coefficients

This notebook computes the residual norm coefficients as part of the variable weights.

In [1]:
import os
import yaml
import copy
import numpy as np
import xarray as xr

In [2]:
from scipy.stats import gmean

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

## WRF

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [6]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/C404_8km/'
ds_example = xr.open_zarr(base_dir+'C404_8km_2000.zarr')
level = np.array(ds_example['bottom_top'])

In [7]:
# # get variable names
# varnames = list(conf['residual'].keys())
# varnames = varnames[:-5] # remove save_loc and others

# varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot', 'WRF_Q_tot_05']
# varname_surf = list(set(varnames) - set(varname_upper))

In [8]:
varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05']
varname_surf = ['WRF_SP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 'WRF_V10', 'WRF_PWAT_05', 'WRF_precip_025']

In [9]:
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]
    
for varname in varname_upper:
    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std
        
    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [10]:
# separate upper air (list) and surf (float) std values
N_upper = len(varname_upper)
std_val_all = list(STD_values.values())
std_val_surf = np.array(std_val_all[:-N_upper])
std_val_upper = std_val_all[-N_upper:]

# combine
std_concat = np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

In [11]:
ds_std = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname)
        ds_std[varname] = data_array

In [12]:
ds_std.to_netcdf(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/C404_residual_TC_1980_2019_12lev.nc')

In [13]:
ds_GP = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/C404_residual_TC_1980_2019_12lev.nc')

ds_full = xr.open_dataset(
    '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_residual_1980_2019_12lev.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== WRF_SP ===================
0.2322716865213044
0.09646890892812135
=================== WRF_T2 ===================
0.793374499808201
1.038070335061416
=================== WRF_TD2 ===================
0.6306996576868847
0.9816457481303755
=================== WRF_U10 ===================
2.240648780027875
3.644687655739152
=================== WRF_V10 ===================
2.4456127790817868
3.466621659562372
=================== WRF_PWAT_05 ===================
0.9760562832232903
1.1418341945303723
=================== WRF_P ===================
[0.23222206 0.2330964  0.23456173 0.23686622 0.2406842  0.24537849
 0.24940171 0.25181831 0.25267363 0.25462248 0.25748538 0.26975701]
[0.09642607 0.09652275 0.0967562  0.09724713 0.09822541 0.09959634
 0.10082328 0.10151499 0.10175701 0.10226155 0.10308294 0.10689739]
=================== WRF_U ===================
[2.31745334 2.20040544 1.97836365 1.88780551 1.84411093 1.61030979
 1.40587841 1.27580823 1.17753212 0.97271956 0.71116441 0